# MeatGuard II — 03 · Test evaluation of all models

Runs every trained model once on the held-out test sessions (`src/evaluate.py`): the selected teacher, the
lightweight baselines, MeatGuard-S without distillation, distilled, pruned and the INT8 TFLite model.

**Saved to `runs/results/`:** `test_results.json` / `.csv` (accuracy, macro-F1, spoiled recall, ROC-AUC, confusion
matrices, 95% CIs from resampling test sessions, parameters, MACs, file size, single-thread CPU latency, McNemar
tests against distilled MeatGuard-S) and `predictions.csv` (P(spoiled) for every model and test image).

CPU latency here is only a reference on the Colab machine; latency on phones is measured in the Android app.

## 1. Setup
Upload the whole `MeatGuard_II` folder to Google Drive and the data zip (`crops/`, `masks/`, `sessions.csv`) next to
it, then edit the two paths below if yours differ. Results are written to `MeatGuard_II/runs/` on Drive, so they
survive disconnects; re-running a training cell resumes from the last saved epoch.

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# --- paths: edit these -------------------------------------------------------------------------
REPO_DIR = "/content/drive/MyDrive/MeatGuard_II" if IN_COLAB else os.path.abspath("..")   # the codebase on Drive
DATA_SRC = "/content/drive/MyDrive/MeatGuard/full.zip" if IN_COLAB else "data/processed/full"  # zip or folder: crops/, masks/, sessions.csv
# -------------------------------------------------------------------------------------------------
DATA_ROOT = "/content/data/full" if IN_COLAB else DATA_SRC     # local copy on the Colab VM (Drive is slow for many small files)
RUNS = os.path.join(REPO_DIR, "runs_specimen")    # results of the specimen-level split (old split: runs/)
WORKERS = 4 if IN_COLAB else 0
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
if IN_COLAB:
    !pip -q install torch-pruning shap onnx onnxruntime ai_edge_litert
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU visible"
import torch, torchvision
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| CUDA", torch.cuda.is_available())

## 2. Data

In [ ]:
# Copy the data to the VM once, then check it: every session in one partition, every listed image present
import zipfile, shutil
import pandas as pd
from pathlib import Path
if IN_COLAB and not os.path.exists(os.path.join(DATA_ROOT, "sessions.csv")):
    if DATA_SRC.endswith(".zip"):
        with zipfile.ZipFile(DATA_SRC) as z:
            z.extractall(DATA_ROOT)
    else:
        shutil.copytree(DATA_SRC, DATA_ROOT)
if not os.path.exists(os.path.join(DATA_ROOT, "sessions.csv")):      # zip made from the parent folder
    nested = [p.parent for p in Path(DATA_ROOT).glob("*/sessions.csv")]
    DATA_ROOT = str(nested[0]) if nested else DATA_ROOT
CROPS, MASKS = (os.path.join(DATA_ROOT, d) for d in ("crops", "masks"))
# Specimen-level split (src.data.specimen_groups): sessions showing the same piece of meat are kept together
SPLIT = os.path.join(REPO_DIR, "runs", "specimen_check", "specimens.csv")
assert os.path.exists(SPLIT), f"missing {SPLIT}"
os.makedirs(RUNS, exist_ok=True)
print("split file:", SPLIT, "| results:", RUNS)

from src.train_teacher import check_split
table = pd.read_csv(SPLIT)
present = {p.relative_to(CROPS).with_suffix("").as_posix() for p in Path(CROPS).rglob("*.jpg")}
table["present"] = table["path"].str.replace(r"\.[^.]+$", "", regex=True).isin(present)
print("split counts:", check_split(SPLIT), "| masks/ present:", os.path.isdir(MASKS))
print(table.groupby(["split", "class"])["present"].agg(["sum", "count"]))
assert table["present"].all(), "Some images listed in sessions.csv are missing from crops/"

## 3. Evaluate

In [ ]:
!python -m src.evaluate --data {CROPS} --split-file {SPLIT} --runs {RUNS} --num-workers {WORKERS}
import json
res = json.load(open(os.path.join(RUNS, "results", "test_results.json")))
cols = ["accuracy", "macro_f1", "recall_positive", "roc_auc", "params_m", "gmacs", "file_size_mb", "cpu_latency_ms"]
display(pd.DataFrame(res["models"]).T[cols].astype(float).round(4))
display(pd.DataFrame(res["mcnemar"]))

In [ ]:
# Confusion matrices (rows = true Fresh, Spoiled; columns = predicted)
import matplotlib.pyplot as plt, numpy as np
names = list(res["models"])
fig, axes = plt.subplots(1, len(names), figsize=(3.2 * len(names), 3))
for ax, n in zip(np.atleast_1d(axes), names):
    cm = np.array(res["models"][n]["confusion"])
    ax.imshow(cm, cmap="Blues")
    for (i, j), v in np.ndenumerate(cm):
        ax.text(j, i, v, ha="center", va="center")
    ax.set_title(n, fontsize=8); ax.set_xticks([0, 1], ["F", "S"]); ax.set_yticks([0, 1], ["F", "S"])
plt.tight_layout(); plt.savefig(os.path.join(RUNS, "results", "confusion_matrices.png"), dpi=200); plt.show()

In [ ]:
# Bundle all result files (JSON/CSV, no weights) into one zip to share: runs/results_bundle.zip on Drive
!python -m src.collect_results --runs {RUNS} --out {RUNS}/results_bundle.zip